# Passo 3: Operações DML no Delta Lake
Aplicamos INSERT, UPDATE e DELETE na tabela Delta na camada Bronze.

In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# Inicializa o Spark
spark = SparkSession.builder \
    .appName("DML-Delta-VAR") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

bronze_path = "s3a://bronze/eventos_delta"
delta_table = DeltaTable.forPath(spark, bronze_path)

print("Placar Inicial:")
delta_table.toDF().orderBy("id_evento").show()

# 1. Cenário VAR 1: Corrigindo o autor do gol (id_evento = 1)
print(">>> VAR analisando o primeiro gol...")
delta_table.update(
    condition="id_evento = 1",
    set={"id_jogador": "99", "revisado_var": "True"}
)

# 2. Cenário VAR 2: Anulando o gol por impedimento (id_evento = 3)
print(">>> VAR anulando o segundo gol por impedimento...")
delta_table.delete(condition="id_evento = 3")

# 3. Cenário Extra: Inserindo um novo evento no apagar das luzes (INSERT / Append)
print(">>> Inserindo o gol do título aos 45 do segundo tempo...")
novo_evento = [(4, 101, 7, "Gol", 90, False)]
colunas = ["id_evento", "id_partida", "id_jogador", "tipo_evento", "minuto_jogo", "revisado_var"]
df_novo = spark.createDataFrame(novo_evento, colunas)
df_novo.write.format("delta").mode("append").save(bronze_path)


print("Placar Final atualizado pelo VAR:")
delta_table.toDF().orderBy("id_evento").show()

print(">>> Consultando a Versão 0 (Antes da intervenção do VAR):")
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(bronze_path)
df_v0.orderBy("id_evento").show()

print("Histórico de versões (Time Travel):")
delta_table.history().select("version", "operation", "timestamp").show(truncate=False)
